In [2]:
!pip install transformers torch pandas newsapi-python

In [3]:
import pandas as pd
from newsapi import NewsApiClient
from transformers import pipeline

EMOTION_TO_MOOD = {
    "joy":      "happy",
    "surprise": "curious",
    "neutral":  "calm",
    "sadness":  "sad",
    "fear":     "anxious",
    "anger":    "angry",
    "disgust":  "angry",
}

# --- CONFIGURATION ---
NEWS_API_KEY = "7e643fe1771e4cebba806bb303eafec3" # <--- Put your key inside the quotes

print("1. Loading the j-hartmann AI Model (This takes a moment)...")
# device=0 forces the model to use the fast Colab GPU
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=0
)

print("2. Connecting to NewsAPI...")
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

# Fetching a massive variety of news using broad categories
articles_response = newsapi.get_everything(
    q='world OR politics OR business OR technology OR science OR health OR sports OR entertainment',
    language='en',
    page_size=100
)
articles_data = articles_response['articles']

print(f"Fetched {len(articles_data)} articles. Starting emotion analysis...")

processed_news = []

for article in articles_data:
    # Skip articles that are broken or missing descriptions
    if not article.get('title') or not article.get('description'):
        continue

    try:
        # The AI reads the description and outputs the emotion
        text_to_analyze = article['description']
        result = emotion_classifier(text_to_analyze)[0]

        # Structure this EXACTLY like your MySQL diagram
        news_row = {
            "title": article['title'],
            "description": article['description'],
            "url": article['url'],
            "source_name": article['source']['name'],
            "published_at": article['publishedAt'],
            "emotion_label": result['label'].lower(),
            "emotion_score": round(result['score'], 4),
            "mood":          EMOTION_TO_MOOD.get(result['label'].lower(), "calm")
        }
        processed_news.append(news_row)

    except Exception as e:
        continue

print("3. Saving the data...")
df = pd.DataFrame(processed_news)
df.to_csv("news_emotion_data.csv", index=False)
print("✅ Done! Your data is formatted and saved as 'news_emotion_data.csv'")

1. Loading the j-hartmann AI Model (This takes a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2. Connecting to NewsAPI...
Fetched 88 articles. Starting emotion analysis...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


3. Saving the data...
✅ Done! Your data is formatted and saved as 'news_emotion_data.csv'


In [4]:
import pandas as pd

# Read the CSV file you just created
df = pd.read_csv("news_emotion_data.csv")

# Display the first 20 articles as a clean table
display(df.head(20))

# Print the total number of articles you successfully grabbed
print(f"\nTotal articles ready for your database: {len(df)}")

,title,description,url,source_name,published_at,emotion_label,emotion_score,mood
0,Trump fires the entire National Science Board,Multiple sources are reporting that the Trump ...,https://www.theverge.com/science/918769/trump-...,The Verge,2026-04-25T19:20:15Z,sadness,0.4244,sad
1,The Next Alzheimer’s Breakthrough Will Take Mo...,"At WIRED Health, pioneering Alzheimer's resear...",https://www.wired.com/story/john-hardy-dementi...,Wired,2026-05-01T18:55:56Z,neutral,0.9204,calm
2,"Players from the NBA, NFL, and MLB call for a ...","The unions backing professional NBA, NFL, MLB,...",https://www.theverge.com/entertainment/922244/...,The Verge,2026-05-01T16:57:27Z,neutral,0.4478,calm
3,Hopeful Thought of the Day: Robots Can’t Do Hu...,"We kill each other just fine, thank you very m...",https://gizmodo.com/hopeful-thought-of-the-day...,Gizmodo.com,2026-04-25T14:39:57Z,neutral,0.4389,calm
4,Your guide to sci-fi streaming season,"I haven't quite figured out the reason why, bu...",https://www.theverge.com/tech/921610/sci-fi-st...,The Verge,2026-05-01T12:25:59Z,neutral,0.7841,calm
5,An expert on Iranian politics reviews the stat...,NPR's Elissa Nadworny talks to Mehrzad Borouje...,https://www.npr.org/2026/05/09/nx-s1-5814972/a...,NPR,2026-05-09T11:43:02Z,neutral,0.8951,calm
6,Get Ready for More Brain-Scanning Consumer Gad...,"Neurable, which makes noninvasive brain-comput...",https://www.wired.com/story/get-ready-for-more...,Wired,2026-04-28T13:00:00Z,neutral,0.9198,calm
7,Why Labour's London squeeze exposes a fragment...,London offers an insight into the dilemma Labo...,https://www.bbc.com/news/articles/cwy2vlww2ezo,BBC News,2026-05-01T06:09:47Z,neutral,0.8570,calm
8,Larry’s risky business,If you want to know whether the AI bubble is b...,https://www.theverge.com/ai-artificial-intelli...,The Verge,2026-04-29T13:57:16Z,neutral,0.9328,calm
9,Netflix’s new INKubator studio will produce an...,"This is Lowpass by Janko Roettgers, a newslett...",https://www.theverge.com/column/930118/netflix...,The Verge,2026-05-13T20:18:45Z,neutral,0.8908,calm



Total articles ready for your database: 88


In [5]:
from google.colab import files

files.download('/content/news_emotion_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>